In [1]:
# Cell 1 — Imports and path setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd

from src.common.utils.config import load_config
from src.modeling.model_io   import load_model
from src.modeling.feature_selector import get_feature_columns
from src.modeling.splitter   import time_split
from src.modeling.metrics    import mape, wmape
from src.modeling.two_stage  import predict_two_stage

print('Imports OK')

Imports OK


In [2]:
# Cell 2 — Config and path constants
cfg     = load_config()
mod_cfg = cfg["modeling"]

MODELS     = Path("../data/output/models")
CELL_DATA  = Path("../data/Intermediate")
THRESHOLD  = mod_cfg["classifier_threshold"]

print(f"Models dir : {MODELS.resolve()}")
print(f"Cell data  : {CELL_DATA.resolve()}")
print(f"Threshold  : {THRESHOLD}")
print(f"Train end  : {mod_cfg['train_end']}  |  Eval start: {mod_cfg['eval_start']}")

Models dir : /Users/mohamedinas/Desktop/SE_projects/8_stax_interview/8_interview_prep/data/output/models
Cell data  : /Users/mohamedinas/Desktop/SE_projects/8_stax_interview/8_interview_prep/data/Intermediate
Threshold  : 0.5
Train end  : 2018-12  |  Eval start: 2019-01


In [3]:
# Cell 3 — Artifact availability check
# All 11 pkl files and all 9 cell parquets must exist before any test runs.

EXPECTED_PKLS = [
    "model_0_power_continuous.pkl",
    "model_1_power_intermittent.pkl",
    "model_2_power_lumpy.pkl",
    "model_3_high-value_active_continuous.pkl",
    "model_4_high-value_active_intermittent.pkl",
    "model_5_high-value_active_lumpy_stage1_clf.pkl",
    "model_5_high-value_active_lumpy_stage2_reg.pkl",
    "model_6_low-value_sporadic_continuous.pkl",
    "model_7_low-value_sporadic_intermittent.pkl",
    "model_8_low-value_sporadic_lumpy_stage1_clf.pkl",
    "model_8_low-value_sporadic_lumpy_stage2_reg.pkl",
]

EXPECTED_CELLS = [
    "cell_0_power_continuous.parquet",
    "cell_1_power_intermittent.parquet",
    "cell_2_power_lumpy.parquet",
    "cell_3_high-value_active_continuous.parquet",
    "cell_4_high-value_active_intermittent.parquet",
    "cell_5_high-value_active_lumpy.parquet",
    "cell_6_low-value_sporadic_continuous.parquet",
    "cell_7_low-value_sporadic_intermittent.parquet",
    "cell_8_low-value_sporadic_lumpy.parquet",
]

all_ok = True
print("=== Model pkl files ===")
for name in EXPECTED_PKLS:
    exists = (MODELS / name).exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status:7s}] {name}")
    if not exists: all_ok = False

print("\n=== Cell cache parquets ===")
for name in EXPECTED_CELLS:
    exists = (CELL_DATA / name).exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status:7s}] {name}")
    if not exists: all_ok = False

print(f"\n{'All artifacts present — proceed.' if all_ok else 'STOP: missing files above must be resolved first.'}")
assert all_ok, "Missing artifact(s) — check output above"

=== Model pkl files ===
  [OK     ] model_0_power_continuous.pkl
  [OK     ] model_1_power_intermittent.pkl
  [OK     ] model_2_power_lumpy.pkl
  [OK     ] model_3_high-value_active_continuous.pkl
  [OK     ] model_4_high-value_active_intermittent.pkl
  [OK     ] model_5_high-value_active_lumpy_stage1_clf.pkl
  [OK     ] model_5_high-value_active_lumpy_stage2_reg.pkl
  [OK     ] model_6_low-value_sporadic_continuous.pkl
  [OK     ] model_7_low-value_sporadic_intermittent.pkl
  [OK     ] model_8_low-value_sporadic_lumpy_stage1_clf.pkl
  [OK     ] model_8_low-value_sporadic_lumpy_stage2_reg.pkl

=== Cell cache parquets ===
  [OK     ] cell_0_power_continuous.parquet
  [OK     ] cell_1_power_intermittent.parquet
  [OK     ] cell_2_power_lumpy.parquet
  [OK     ] cell_3_high-value_active_continuous.parquet
  [OK     ] cell_4_high-value_active_intermittent.parquet
  [OK     ] cell_5_high-value_active_lumpy.parquet
  [OK     ] cell_6_low-value_sporadic_continuous.parquet
  [OK     ] cell_7_l

In [4]:
# Cell 4 — Test runner helpers

def load_and_split(cell_name):
    """Load cached cell parquet and return (df_train, df_eval, feature_cols)."""
    df = pd.read_parquet(CELL_DATA / cell_name)
    df_train, df_eval = time_split(df, mod_cfg["train_end"], mod_cfg["eval_start"])
    feature_cols, _ = get_feature_columns(
        df_train, mod_cfg["target_regressor"], mod_cfg["drop_columns"]
    )
    return df_train, df_eval, feature_cols


def report(label, y_true, y_pred):
    """Print MAPE and WMAPE and return as dict."""
    mape_val, zero_frac = mape(y_true, y_pred)
    wmape_val = wmape(y_true, y_pred)
    print(f"  MAPE  : {mape_val:7.2f}%  (zero actuals excluded: {zero_frac:.1%})")
    print(f"  WMAPE : {wmape_val:7.2f}%")
    return {"label": label, "mape": mape_val, "wmape": wmape_val, "zero_frac": zero_frac}


RESULTS = []  # accumulate across all cells
print("Helpers ready")

Helpers ready


In [5]:
# Cell 5 — Grid cell 0: Power + Continuous  (single-stage regressor)
print("Grid cell 0 — Power + Continuous")

reg = load_model(MODELS, "model_0_power_continuous")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_0_power_continuous.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("Power + Continuous", y_true, y_pred))

Grid cell 0 — Power + Continuous
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 114,889 | Feature cols: 41
  MAPE  :   73.55%  (zero actuals excluded: 80.0%)
  WMAPE :  143.89%


In [6]:
# Cell 6 — Grid cell 1: Power + Intermittent  (single-stage regressor)
print("Grid cell 1 — Power + Intermittent")

reg = load_model(MODELS, "model_1_power_intermittent")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_1_power_intermittent.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("Power + Intermittent", y_true, y_pred))

Grid cell 1 — Power + Intermittent
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 510,247 | Feature cols: 41
  MAPE  :   88.22%  (zero actuals excluded: 80.1%)
  WMAPE :  145.34%


In [7]:
# Cell 7 — Grid cell 2: Power + Lumpy  (single-stage regressor)
print("Grid cell 2 — Power + Lumpy")

reg = load_model(MODELS, "model_2_power_lumpy")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_2_power_lumpy.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("Power + Lumpy", y_true, y_pred))

Grid cell 2 — Power + Lumpy
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 98,194 | Feature cols: 41
  MAPE  :  110.32%  (zero actuals excluded: 86.6%)
  WMAPE :  156.31%


In [8]:
# Cell 8 — Grid cell 3: High-Value Active + Continuous  (single-stage regressor)
print("Grid cell 3 — High-Value Active + Continuous")

reg = load_model(MODELS, "model_3_high-value_active_continuous")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_3_high-value_active_continuous.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("High-Value Active + Continuous", y_true, y_pred))

Grid cell 3 — High-Value Active + Continuous
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 551,118 | Feature cols: 41
  MAPE  :   70.13%  (zero actuals excluded: 85.1%)
  WMAPE :  160.63%


In [9]:
# Cell 9 — Grid cell 4: High-Value Active + Intermittent  (single-stage regressor)
print("Grid cell 4 — High-Value Active + Intermittent")

reg = load_model(MODELS, "model_4_high-value_active_intermittent")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_4_high-value_active_intermittent.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("High-Value Active + Intermittent", y_true, y_pred))

Grid cell 4 — High-Value Active + Intermittent
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 1,675,870 | Feature cols: 41
  MAPE  :   78.83%  (zero actuals excluded: 90.5%)
  WMAPE :  205.42%


In [10]:
# Cell 10 — Grid cell 5: High-Value Active + Lumpy  (two-stage: classifier + regressor)
print("Grid cell 5 — High-Value Active + Lumpy  [TWO-STAGE]")

clf = load_model(MODELS, "model_5_high-value_active_lumpy_stage1_clf")
reg = load_model(MODELS, "model_5_high-value_active_lumpy_stage2_reg")
print(f"  Stage 1: {type(clf).__name__} | Stage 2: {type(reg).__name__}")
print(f"  Threshold: {THRESHOLD}")

_, df_eval, feature_cols = load_and_split("cell_5_high-value_active_lumpy.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = predict_two_stage(clf, reg, X_eval, THRESHOLD)

nonzero_predicted = (y_pred > 0).mean()
print(f"  Stage 1 fired (predicted nonzero): {nonzero_predicted:.1%} of eval rows")

RESULTS.append(report("High-Value Active + Lumpy", y_true, y_pred))

Grid cell 5 — High-Value Active + Lumpy  [TWO-STAGE]
  Stage 1: XGBClassifier | Stage 2: XGBRegressor
  Threshold: 0.5
  Eval rows: 231,892 | Feature cols: 41
  Stage 1 fired (predicted nonzero): 30.1% of eval rows
  MAPE  :  192.22%  (zero actuals excluded: 93.6%)
  WMAPE :  826.17%


In [11]:
# Cell 11 — Grid cell 6: Low-Value Sporadic + Continuous  (single-stage regressor)
print("Grid cell 6 — Low-Value Sporadic + Continuous")

reg = load_model(MODELS, "model_6_low-value_sporadic_continuous")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_6_low-value_sporadic_continuous.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("Low-Value Sporadic + Continuous", y_true, y_pred))

Grid cell 6 — Low-Value Sporadic + Continuous
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 352,429 | Feature cols: 41
  MAPE  :   76.75%  (zero actuals excluded: 91.6%)
  WMAPE :  200.19%


In [12]:
# Cell 12 — Grid cell 7: Low-Value Sporadic + Intermittent  (single-stage regressor)
print("Grid cell 7 — Low-Value Sporadic + Intermittent")

reg = load_model(MODELS, "model_7_low-value_sporadic_intermittent")
print(f"  Model type: {type(reg).__name__} | Features trained on: {reg.n_features_in_}")

_, df_eval, feature_cols = load_and_split("cell_7_low-value_sporadic_intermittent.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = np.clip(reg.predict(X_eval), 0, None)

RESULTS.append(report("Low-Value Sporadic + Intermittent", y_true, y_pred))

Grid cell 7 — Low-Value Sporadic + Intermittent
  Model type: XGBRegressor | Features trained on: 41
  Eval rows: 807,960 | Feature cols: 41
  MAPE  :   87.88%  (zero actuals excluded: 96.7%)
  WMAPE :  312.65%


In [13]:
# Cell 13 — Grid cell 8: Low-Value Sporadic + Lumpy  (two-stage: classifier + regressor)
print("Grid cell 8 — Low-Value Sporadic + Lumpy  [TWO-STAGE]")

clf = load_model(MODELS, "model_8_low-value_sporadic_lumpy_stage1_clf")
reg = load_model(MODELS, "model_8_low-value_sporadic_lumpy_stage2_reg")
print(f"  Stage 1: {type(clf).__name__} | Stage 2: {type(reg).__name__}")
print(f"  Threshold: {THRESHOLD}")

_, df_eval, feature_cols = load_and_split("cell_8_low-value_sporadic_lumpy.parquet")
print(f"  Eval rows: {len(df_eval):,} | Feature cols: {len(feature_cols)}")

X_eval = df_eval[feature_cols].values
y_true = df_eval[mod_cfg["target_regressor"]].values
y_pred = predict_two_stage(clf, reg, X_eval, THRESHOLD)

nonzero_predicted = (y_pred > 0).mean()
print(f"  Stage 1 fired (predicted nonzero): {nonzero_predicted:.1%} of eval rows")

RESULTS.append(report("Low-Value Sporadic + Lumpy", y_true, y_pred))

Grid cell 8 — Low-Value Sporadic + Lumpy  [TWO-STAGE]
  Stage 1: XGBClassifier | Stage 2: XGBRegressor
  Threshold: 0.5
  Eval rows: 97,899 | Feature cols: 41
  Stage 1 fired (predicted nonzero): 20.1% of eval rows
  MAPE  :  122.25%  (zero actuals excluded: 97.9%)
  WMAPE : 1486.47%


In [14]:
# Cell 14 — Summary results table
summary = pd.DataFrame(RESULTS).set_index("label")
summary["mape"]      = summary["mape"].map("{:.2f}%".format)
summary["wmape"]     = summary["wmape"].map("{:.2f}%".format)
summary["zero_frac"] = summary["zero_frac"].map("{:.1%}".format)
summary.columns      = ["MAPE", "WMAPE", "Zero-actual fraction"]

print("\n=== Prediction test — all 9 grid cells ===")
print(summary.to_string())
print(f"\nAll {len(RESULTS)}/9 models loaded and predicted successfully.")


=== Prediction test — all 9 grid cells ===
                                      MAPE     WMAPE Zero-actual fraction
label                                                                    
Power + Continuous                  73.55%   143.89%                80.0%
Power + Intermittent                88.22%   145.34%                80.1%
Power + Lumpy                      110.32%   156.31%                86.6%
High-Value Active + Continuous      70.13%   160.63%                85.1%
High-Value Active + Intermittent    78.83%   205.42%                90.5%
High-Value Active + Lumpy          192.22%   826.17%                93.6%
Low-Value Sporadic + Continuous     76.75%   200.19%                91.6%
Low-Value Sporadic + Intermittent   87.88%   312.65%                96.7%
Low-Value Sporadic + Lumpy         122.25%  1486.47%                97.9%

All 9/9 models loaded and predicted successfully.
